# Notebook 2 — EDA & Dataset Analysis
**BRFSS 2022 | Heart Disease Prediction Thesis**

Mục tiêu: Khám phá dữ liệu sau clean, phân tích phân bố, visualize insights cho report/slide.

In [ ]:
!pip install pandas numpy matplotlib seaborn -q


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

print("Libraries loaded ✓")


In [ ]:
## 1. Upload Dataset from Your Computer

import os
from google.colab import files

# ── Tạo thư mục lưu output
OUTPUT_DIR = '/content/thesis'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

# ── Upload file từ máy tính
uploaded = files.upload()

# Lấy tên file vừa upload
filename = list(uploaded.keys())[0]
RAW_PATH = f'/content/{filename}'

# ── Load raw data
df_raw = pd.read_csv(RAW_PATH, low_memory=False)

print(f"Uploaded file: {filename}")
print(f"Raw shape: {df_raw.shape}")
print(f"Rows: {df_raw.shape[0]:,}  |  Columns: {df_raw.shape[1]}")

Output directory: /content/thesis


TypeError: 'NoneType' object is not subscriptable

## 1. Dataset Overview

In [ ]:
# ── Basic info
print("="*55)
print(f"  Total Records    : {len(df_raw):,}")
print(f"  Total Features   : {df_raw.shape[1]-1}")
print(f"  CVD Cases (=1)   : {df_raw['CVD'].sum():,} ({df_raw['CVD'].mean()*100:.1f}%)")
print(f"  Non-CVD (=0)     : {(df_raw['CVD']==0).sum():,} ({(1-df_raw['CVD'].mean())*100:.1f}%)")
print(f"  Missing Values   : {df_raw.isnull().sum().sum()}")
print("="*55)


In [ ]:
# ── Descriptive statistics
print("Descriptive statistics — Numeric features:")
numeric_cols = ['PHYSHLTH','MENTHLTH','SLEPTIM1','INCOME3']
df_raw[numeric_cols + ['CVD']].describe().round(2)


## 2. Class Imbalance Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Count
counts = df_raw['CVD'].value_counts().sort_index()
bars = axes[0].bar(['Non-CVD (0)', 'CVD (1)'], counts.values,
                   color=['#2196F3','#F44336'], width=0.5, edgecolor='white')
axes[0].set_title('Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000,
                 f'{val:,}', ha='center', fontsize=10)

# Percentage
pcts = counts / counts.sum() * 100
axes[1].pie(pcts.values, labels=['Non-CVD','CVD'],
            colors=['#2196F3','#F44336'],
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor':'white','linewidth':2})
axes[1].set_title('Class Ratio (%)', fontsize=13, fontweight='bold')

# Imbalance ratio
ratio = counts[0] / counts[1]
axes[2].barh(['Imbalance Ratio'], [ratio], color='#FF9800', edgecolor='white')
axes[2].axvline(x=1, color='gray', linestyle='--', alpha=0.7)
axes[2].set_title(f'Imbalance Ratio = {ratio:.1f}:1', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Non-CVD : CVD')
axes[2].text(ratio/2, 0, f'{ratio:.1f}x', ha='center', va='center',
             color='white', fontweight='bold', fontsize=13)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/nb2_class_imbalance.png', dpi=150, bbox_inches='tight')
plt.show()


## 3. Age Distribution by CVD

In [ ]:
age_labels = {
    1:'18-24', 2:'25-29', 3:'30-34', 4:'35-39', 5:'40-44',
    6:'45-49', 7:'50-54', 8:'55-59', 9:'60-64', 10:'65-69',
    11:'70-74', 12:'75-79', 13:'80+', 14:'Unknown'
}
df_raw['AGE_LABEL'] = df_raw['_AGEG5YR'].map(age_labels)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count by age & CVD
age_cvd = df.groupby(['AGE_LABEL','CVD']).size().unstack(fill_value=0)
age_order = [age_labels[i] for i in range(1,14)]
age_cvd = age_cvd.reindex(age_order)

age_cvd.plot(kind='bar', ax=axes[0], color=['#2196F3','#F44336'],
             edgecolor='white', width=0.7)
axes[0].set_title('CVD Count by Age Group', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Age Group')
axes[0].set_ylabel('Count')
axes[0].legend(['Non-CVD','CVD'])
axes[0].tick_params(axis='x', rotation=45)

# CVD rate by age
age_rate = df.groupby('AGE_LABEL')['CVD'].mean().reindex(age_order) * 100
axes[1].plot(range(len(age_order)), age_rate.values, 'o-',
             color='#F44336', linewidth=2, markersize=7)
axes[1].fill_between(range(len(age_order)), age_rate.values,
                     alpha=0.15, color='#F44336')
axes[1].set_xticks(range(len(age_order)))
axes[1].set_xticklabels(age_order, rotation=45, ha='right')
axes[1].set_title('CVD Rate (%) by Age Group', fontsize=13, fontweight='bold')
axes[1].set_ylabel('CVD Rate (%)')
axes[1].set_xlabel('Age Group')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/nb2_age_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. BMI, Smoking, Diabetes, Key Risk Factors

In [ ]:
# ── CVD rate by key categorical features
features_to_plot = {
    '_BMI5CAT':  {1:'Underweight', 2:'Normal', 3:'Overweight', 4:'Obese'},
    'SMOKE100':  {1:'Yes', 2:'No'},
    'DIABETE4':  {1:'Yes', 2:'Preg.', 3:'No', 4:'Pre-diab'},
    'EXERANY2':  {1:'Active', 2:'Inactive'},
    'CHCKDNY2':  {1:'Yes', 2:'No'},
    'ADDEPEV3':  {1:'Yes', 2:'No'},
    'DIFFWALK':  {1:'Yes', 2:'No'},
    'GENHLTH':   {1:'Excellent', 2:'Very Good', 3:'Good', 4:'Fair', 5:'Poor'},
}

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for idx, (col, label_map) in enumerate(features_to_plot.items()):
    ax = axes[idx]
    temp = df_raw[df_raw[col].isin(label_map.keys())].copy()
    temp['Label'] = temp[col].map(label_map)
    rate = temp.groupby('Label')['CVD'].mean() * 100
    rate = rate.reindex(list(label_map.values())).dropna()

    colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.9, len(rate)))
    bars = ax.bar(rate.index, rate.values, color=colors, edgecolor='white', width=0.6)
    ax.set_title(col, fontsize=11, fontweight='bold')
    ax.set_ylabel('CVD Rate (%)')
    ax.tick_params(axis='x', rotation=30)
    for bar, val in zip(bars, rate.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=9)

plt.suptitle('CVD Rate (%) by Key Risk Factors', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/nb2_risk_factors.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Correlation Heatmap

In [ ]:
# ── Numeric + binary features correlation với CVD
corr_cols = [
    'PHYSHLTH','MENTHLTH','SLEPTIM1','INCOME3',
    'GENHLTH','DIABETE4','SMOKE100','EXERANY2',
    'CHCKDNY2','ADDEPEV3','DIFFWALK','HAVARTH4','CVD'
]

corr_df = df_raw[[c for c in corr_cols if c in df_raw.columns]].copy()
corr_matrix = corr_df.corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, ax=ax,
            linewidths=0.5, annot_kws={'size': 9},
            vmin=-1, vmax=1)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/nb2_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Physical & Mental Health Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, col, title in zip(axes,
    ['PHYSHLTH', 'MENTHLTH'],
    ['Physical Health (Days Not Good)', 'Mental Health (Days Not Good)']):

    cvd0 = df_raw[df_raw['CVD']==0][col]
    cvd1 = df_raw[df_raw['CVD']==1][col]

    ax.hist(cvd0, bins=30, alpha=0.6, color='#2196F3', label='Non-CVD', density=True)
    ax.hist(cvd1, bins=30, alpha=0.6, color='#F44336', label='CVD', density=True)
    ax.axvline(cvd0.median(), color='#1565C0', linestyle='--', label=f'Median Non-CVD: {cvd0.median():.0f}')
    ax.axvline(cvd1.median(), color='#B71C1C', linestyle='--', label=f'Median CVD: {cvd1.median():.0f}')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Days per Month')
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/nb2_health_days.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Gender & CVD

In [ ]:
sex_labels = {1: 'Male', 2: 'Female'}
df_raw['SEX_LABEL'] = df_raw['SEXVAR'].map(sex_labels)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Count
sex_cvd = df.groupby(['SEX_LABEL','CVD']).size().unstack(fill_value=0)
sex_cvd.plot(kind='bar', ax=axes[0], color=['#2196F3','#F44336'],
             edgecolor='white', width=0.5)
axes[0].set_title('CVD Count by Gender', fontsize=12, fontweight='bold')
axes[0].set_xlabel('')
axes[0].legend(['Non-CVD','CVD'])
axes[0].tick_params(axis='x', rotation=0)

# Rate
sex_rate = df.groupby('SEX_LABEL')['CVD'].mean() * 100
axes[1].bar(sex_rate.index, sex_rate.values,
            color=['#1976D2','#E91E63'], edgecolor='white', width=0.4)
axes[1].set_title('CVD Rate (%) by Gender', fontsize=12, fontweight='bold')
axes[1].set_ylabel('CVD Rate (%)')
for i, (lab, val) in enumerate(sex_rate.items()):
    axes[1].text(i, val + 0.2, f'{val:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/nb2_gender_cvd.png', dpi=150, bbox_inches='tight')
plt.show()


## 8. Summary Statistics Table

In [ ]:
# ── Generate summary table CVD vs Non-CVD
summary_rows = []
for col, label_map in features_to_plot.items():
    if col not in df_raw.columns:
        continue
    for code, label in label_map.items():
        n_cvd  = ((df_raw['CVD']==1) & (df_raw[col]==code)).sum()
        n_ncvd = ((df_raw['CVD']==0) & (df_raw[col]==code)).sum()
        total  = n_cvd + n_ncvd
        if total > 0:
            summary_rows.append({
                'Feature': col,
                'Category': label,
                'CVD (n)': n_cvd,
                'Non-CVD (n)': n_ncvd,
                'CVD Rate (%)': round(n_cvd/total*100, 1)
            })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))
summary_df.to_csv(f'{OUTPUT_DIR}/nb2_summary_table.csv', index=False)
print("\nSummary table saved ✓")


## ✅ Notebook 2 Complete

**Output plots:**
- `nb2_class_imbalance.png`
- `nb2_age_distribution.png`
- `nb2_risk_factors.png`
- `nb2_correlation_heatmap.png`
- `nb2_health_days.png`
- `nb2_gender_cvd.png`
- `nb2_summary_table.csv`